# 🎯 [TRN01] 드론 시뮬레이터 비행로그 기반 숙련도 분석기
### Google Colab 실습 환경 (교육·훈련 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/TRN01/trn01_m07_solution.ipynb)

---

## 📌 1. 프로젝트 개요 및 학습 목표

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 교육·훈련 (Level 3) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.5시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 학습 목표 및 대상자
러시아-우크라이나 전쟁에서 소형 드론 조종 인력을 단기간에 양성·평가해야 했던 전장 요구를 배경으로, 교육용 가상 비행로그를 직접 생성·분석하며 드론 숙련도 평가 파이프라인 전 주기를 설계·구현하는 프로젝트다. 최종 목표는 교관 평가와 정합성을 갖춘 객관적 지표와 개인별 피드백을 산출하되, 등급과 재교육 결정은 교관이 근거를 기록해 확정하는 신뢰 가능한 분석기를 완성하는 것이다.

#### 본 프로젝트를 통해 익히는 핵심 역량

- **전술 시계열 데이터 설계 및 정제 역량:** 비행로그 스키마와 보안(가명화) 규칙을 정의하고, PWM 정규화·Hampel 필터·보간으로 센서 잡음과 단절을 복구하는 실무 기법을 익힌다.
- **도메인 기반 숙련도 모델링 및 한계 분석 역량:** 경로·호버링 Baseline 채점부터 장애물 회피·비상 착륙 반응 기반 정밀 평가 엔진까지 단계적으로 고도화하며, 혼동표·오탐 분석으로 모델의 사각지대를 찾아내는 방법을 습득한다.
- **통계적 신뢰도 검증 및 조작 습관 진단 역량:** 단발성 점수의 오류를 줄이기 위해 다회차 비행 묶음 평가와 부트스트랩 신뢰구간을 적용하고, 급조작·과보정 등 개인별 습관을 정량적으로 판별하는 원리를 익힌다.
- **HITL(인간-중심) 거버넌스 및 시스템 복원력 설계 역량:** 저신뢰·고위험 사례 우선 검토, 2단계 교관 확인, 해시 체인 감사 로그로 AI 책임 분계를 구현하고, 대량 배치 스트레스 시험과 AAR 재튜닝으로 운용 안정성을 확보하는 역량을 기른다.

#### 추천 학습 대상자 (페르소나)

- **드론·무인기 교육훈련 교관 및 평가 담당자:** 시뮬레이터 비행로그를 정량 지표로 해석하고, 교관의 경험적 평가와 AI 분석 결과를 결합해 개인별 맞춤 피드백·재교육 기준을 정립하고자 하는 현장 실무자.
- **국방 AI·데이터 분석 실무자 및 개발자:** Python 기본 문법을 바탕으로 시계열 노이즈 전처리, 다중 지표 채점 모델링, 부트스트랩 신뢰도 산출, HITL 승인 워크플로우까지 End-to-End 분석 파이프라인을 직접 구현하려는 개발자.
- **무인체계 시뮬레이터·훈련체계 기획 및 품질 검증 담당자:** 로그 스키마 표준화, 대량 배치 스트레스 테스트, 사후강평(AAR) 기반 파라미터 재튜닝, 감사 로그 거버넌스 등 신뢰 가능한 AI 보조 평가 체계의 설계 기준을 익히려는 엔지니어.


In [1]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


실행 환경 준비 완료 (Python 3.14.10)


---

## 🎯 미션 07. 교관이 직접 조종자 성향을 진단하고 피드백을 입력하는 HITL 평가 인터페이스 설계
**소요시간**: 약 3.0시간 | **필수 산출물**: src/hitl/trn01_approval_workflow.py (필수), docs/trn01_hitl_guide.md (필수)

### 📋 미션 개요 (Overview)
AI 숙련도 분석 결과는 교관 판단을 보조하는 참고 지표이며, 재교육 지정이나 자격 보류 같은 인사·훈련 결정은 반드시 사람이 책임지고 확정해야 한다. 본 미션에서는 저신뢰·C등급 제안을 우선 정렬해 검토 대기열에 등록하고, 교관 ID·진단 근거 입력과 불리한 결정 시 2단계 교관 확인을 강제하는 HITL 워크플로우(`trn01_approval_workflow.py`)를 설계한다. 모든 제안·결정 이력을 해시 체인 감사 로그로 보호해 위·변조 검출까지 검증한다.

### 🎓 학습 목표 (Objectives)
AI 판정 결과를 실무 의사결정에 안전하게 연결하기 위한 **HITL(Human-in-the-Loop) 승인 워크플로우와 감사 추적(Audit Trail) 설계 역량**을 익힌다. 저신뢰·C등급 우선 검토 큐 구성, 교관 ID 및 진단 근거 입력 강제, 재교육·자격 보류 등 불리한 처분 시 2단계 교관 확인 절차를 코드로 구현하는 방법과, 해시 체인 감사 로그를 통해 기록 위·변조를 탐지하는 보안·거버넌스 원리를 체득한다.

### 🛠️ 단계별 수행 및 AI 활용 가이드
#### Step 1. 미션7 코드를 실행해 검토 순서와 AI 확정 건수(0건)를 확인한다.
#### 이 단계의 역할
미션7 코드를 실행해 검토 순서와 AI 확정 건수(0건)를 확인한다.

#### 해야 할 일

- 미션7 코드를 실행해 검토 순서와 AI 확정 건수(0건)를 확인한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> AI 자동 확정 0건, 규칙 위반 6건 차단 로깅 성공률 100%, 변조 위치(8번째 기록) 검출

#### 학습자가 제출해야 하는 내용

- 미션7 코드를 실행해 검토 순서와 AI 확정 건수(0건)를 확인한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> AI 제안 결과를 항상 '검토 대기' 상태로 등록하고, 신뢰도 Low 및 C등급 조종자를 최우선으로 정렬하는 교관 검토 대기열 로직을 구현해줘.
**AI 응답 검증 포인트**

- AI가 단독으로 최종 확정한 건수가 0건인지(모든 제안이 대기열을 거치는지) 단언문(assert)으로 검증
- 우선순위 정렬 순서(Low 신뢰·C등급 우선)가 테스트 케이스에서 정확히 동작하는지 확인

#### Step 2. 차단된 규칙 위반 결정 6건과 차단 사유를 정리한다.
#### 이 단계의 역할
차단된 규칙 위반 결정 6건과 차단 사유를 정리한다.

#### 해야 할 일

- 차단된 규칙 위반 결정 6건과 차단 사유를 정리한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> AI 자동 확정 0건, 규칙 위반 6건 차단 로깅 성공률 100%, 변조 위치(8번째 기록) 검출

#### 학습자가 제출해야 하는 내용

- 차단된 규칙 위반 결정 6건과 차단 사유를 정리한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 교관 ID 누락, 진단 근거 미입력, 불리한 결정(재교육/자격 보류) 시 2차 교관 확인 누락 등 6가지 규칙 위반 시도를 차단하고 사유를 반환하는 테스트를 작성해줘.
**AI 응답 검증 포인트**

- 규칙 위반 6건이 예외 처리되거나 차단 상태로 기록되며, 우회 경로가 없는지 확인
- 반환된 차단 사유가 운영자가 이해하고 보완할 수 있을 만큼 명확한지 점검

#### Step 3. 자격 보류의 2단계 확인 과정과 감사 로그 변조 검출 결과를 확인한다.
#### 이 단계의 역할
자격 보류의 2단계 확인 과정과 감사 로그 변조 검출 결과를 확인한다.

#### 해야 할 일

- 자격 보류의 2단계 확인 과정과 감사 로그 변조 검출 결과를 확인한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> AI 자동 확정 0건, 규칙 위반 6건 차단 로깅 성공률 100%, 변조 위치(8번째 기록) 검출

#### 학습자가 제출해야 하는 내용

- 자격 보류의 2단계 확인 과정과 감사 로그 변조 검출 결과를 확인한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 자격 보류 결정의 2단계 교관 확인 흐름과 함께, 모든 제안·결정 이력을 SHA-256 해시 체인 감사 로그에 기록하고 과거 레코드 변조 시 즉시 검출하는 검증 코드를 짜줘.
**AI 응답 검증 포인트**

- 감사 로그 중간 행의 등급이나 근거 텍스트를 1글자 수정했을 때 해시 체인 검증이 변조를 잡아내는지 확인
- 1차 교관과 2차 확인 교관 ID가 동일인일 경우 차단되는지(이해상충 방지) 점검

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
기본 파이프라인 수행 후 심화 튜닝 및 조건 변경을 통해 최적화 역량을 다진다.

#### 해야 할 일
재교육 지정(RETRAIN)에도 두 번째 교관 확인을 요구하도록 바꿀지 토의하고, 교관 업무량과 공정성 측면의 장단점을 정리한다.

#### 학습자가 제출해야 하는 내용

- 파라미터 변경 전후 비교 분석 메모 및 최종 최적화 결과

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 조종자 피드백 양식에 '재교육 훈련 목표'와 '재평가 예정일' 필수 항목을 추가하고, 미입력 시 HITL 확정을 막는 검증 규칙을 확장해줘.
**AI 응답 검증 포인트**

- 정상 입력 시에는 통과하고 신규 필수 항목 누락 시에는 차단되는지 단위 테스트로 확인
- 기존 해시 체인 감사 로그 구조와 호환되는지 점검


In [2]:
"""AI가 제안한 숙련도 등급·조작 성향을 교관이 직접 확인·수정하고 피드백을 입력하는 HITL 평가 인터페이스를 구현한다. AI는 등급을 확정하지 않으며, 불리한 결정(재교육 지정·자격 보류)은 상세 근거와 두 번째 교관 확인을 요구하고, 모든 결정은 위·변조 확인이 가능한 감사 로그에 남긴다."""
import hashlib
import json
import pandas as pd

ACTIONS = {"CONFIRM": "AI 제안 등급 확정", "MODIFY": "등급 수정", "REFLIGHT": "추가 비행 후 재평가", "RETRAIN": "재교육 지정", "QUAL_HOLD": "자격 부여 보류"}
CONF_ORDER = {"Low": 0, "Medium": 1, "High": 2}


class ReviewError(ValueError):
    pass


class FakeClock:
    t = 0.0

    def __call__(self):
        return self.t


class ReviewQueue:
    def __init__(self, clock):
        self.items, self.log, self.clock = {}, [], clock

    def _record(self, body):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {**body, "ts": round(self.clock(), 1)}
        self.log.append({**body, "prev": prev, "hash": hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest()})

    def verify_log(self):
        prev = "GENESIS"
        for n, rec in enumerate(self.log, start=1):
            body = {k: v for k, v in rec.items() if k not in ("prev", "hash")}
            if rec["prev"] != prev or rec["hash"] != hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest():
                return False, n
            prev = rec["hash"]
        return True, -1

    def submit(self, pilot, grade, conf, habits, evidence):
        """AI 제안 등록: 항상 PENDING으로 시작하며 AI가 확정하지 않는다."""
        self.items[pilot] = {"pilot": pilot, "ai_grade": grade, "conf": conf, "habits": habits, "evidence": evidence,
                             "status": "PENDING", "final": None, "submitted": self.clock(), "approvals": []}
        self._record({"event": "AI_SUGGEST", "pilot": pilot, "grade": grade, "conf": conf, "by": "AI"})

    def order(self):
        """검토 순서: 신뢰도 낮은 제안 -> C 등급 -> 나머지(교관이 판단이 필요한 건부터 본다)."""
        todo = [i for i in self.items.values() if i["status"] in ("PENDING", "AWAIT_2ND")]
        return sorted(todo, key=lambda i: (CONF_ORDER[i["conf"]], i["ai_grade"] != "C", i["pilot"]))

    def decide(self, pilot, action, instructor, reason, new_grade=None):
        if action not in ACTIONS:
            raise ReviewError(f"허용되지 않은 조치: {action}")
        if not instructor:
            raise ReviewError("교관 ID 필요")
        it = self.items.get(pilot)
        if it is None or it["status"] == "FINAL":
            raise ReviewError("검토 대기 건이 아님")
        if len(reason.strip()) < 10:
            raise ReviewError("진단·피드백 10자 이상 필요")
        if action == "MODIFY" and new_grade not in ("A", "B", "C"):
            raise ReviewError("수정 등급(A/B/C) 필요")
        if action in ("RETRAIN", "QUAL_HOLD") and len(reason.strip()) < 20:
            raise ReviewError("불리한 결정은 근거 20자 이상 필요")
        if action == "QUAL_HOLD":
            if instructor in it["approvals"]:
                raise ReviewError("두 번째 확인은 다른 교관이 해야 함")
            it["approvals"].append(instructor)
            self._record({"event": "DECIDE", "pilot": pilot, "action": action, "by": instructor, "reason": reason, "step": len(it["approvals"])})
            if len(it["approvals"]) < 2:
                it["status"] = "AWAIT_2ND"
                return "두 번째 교관 확인 대기"
        else:
            self._record({"event": "DECIDE", "pilot": pilot, "action": action, "by": instructor, "reason": reason, "new_grade": new_grade})
        it["status"], it["action"], it["decided"] = "FINAL", action, self.clock()
        it["final"] = new_grade if action == "MODIFY" else (None if action == "REFLIGHT" else it["ai_grade"])
        return "확정"


clock = FakeClock()
rq = ReviewQueue(clock)
# [교육용가상자료] 미션 5 결과에서 넘어온 AI 제안 6건(가명)
for p, g, c, h, e in [("P003", "A", "High", [], "경로 RMSE 하위 10%, 접지 0.4m/s"), ("P011", "C", "High", ["급조작"], "경착륙 2회, 스틱 변화율 상위 5%"),
                      ("P017", "B", "Low", ["과보정"], "비행별 점수 편차 큼(48~79)"), ("P024", "C", "Medium", ["과보정"], "장애물 근접 3회"),
                      ("P030", "B", "High", [], "호버링 안정, 장애물 반응 1.2s"), ("P038", "C", "Low", [], "돌풍 비행만 낮음")]:
    clock.t += 1; rq.submit(p, g, c, h, e)
print("검토 순서:", [f"{i['pilot']}({i['ai_grade']}/{i['conf']})" for i in rq.order()])
print("AI가 확정한 건수:", sum(i["status"] == "FINAL" for i in rq.items.values()))

for args in [("P011", "RETRAIN", "", "경착륙 반복, 착륙 절차 재교육"), ("P003", "CONFIRM", "inst01", "좋음"), ("P017", "MODIFY", "inst01", "편차가 커서 등급 조정함"),
             ("P024", "RETRAIN", "inst02", "근접 3회 재교육"), ("P003", "AUTO", "inst01", "자동 확정 시도합니다")]:
    try:
        rq.decide(*args)
    except ReviewError as e:
        print(f"차단: {args[0]} {args[1]} -> {e}")

clock.t += 120; print("P003:", rq.decide("P003", "CONFIRM", "inst01", "경로·호버링 안정, 돌발 대응 양호 확인"))
clock.t += 200; print("P017:", rq.decide("P017", "REFLIGHT", "inst01", "점수 편차 큼, 주간 비행 2회 추가 후 재평가"))
clock.t += 150; print("P011:", rq.decide("P011", "RETRAIN", "inst02", "경착륙 2회와 급조작 확인, 착륙 감속 절차 재교육 지정"))
clock.t += 90; print("P024 1차:", rq.decide("P024", "QUAL_HOLD", "inst02", "장애물 근접 3회 반복, 회피 기동 숙달 전 자격 보류"))
clock.t += 10
try:
    rq.decide("P024", "QUAL_HOLD", "inst02", "같은 교관이 두 번째 확인까지 직접 하려고 시도함")
except ReviewError as e:
    print("차단: P024 QUAL_HOLD ->", e)
clock.t += 300; print("P024 2차:", rq.decide("P024", "QUAL_HOLD", "inst05", "영상 재확인 결과 회피 지연 동의, 보류 유지"))
clock.t += 60; print("P038:", rq.decide("P038", "MODIFY", "inst03", "돌풍 외 비행은 B 수준, 돌풍 대응만 보완", new_grade="B"))

df = pd.DataFrame(rq.items.values())
df["대기(s)"] = (df["decided"] - df["submitted"]).round(0)
print("\n" + df[["pilot", "ai_grade", "conf", "action", "final", "대기(s)"]].astype(object).fillna("-").to_string(index=False))
fin = df[df["status"] == "FINAL"]
print(f"교관 결정 {len(fin)}건 중 AI 제안 등급과 다른 결정 {int((fin['final'] != fin['ai_grade']).sum())}건(추가 비행 포함), 미검토 {int((df['status'] != 'FINAL').sum())}건은 대기 유지")
print(f"\n감사 로그 {len(rq.log)}건 기록(제안 {len(df)}건 + 결정 {len(rq.log) - len(df)}건, 로깅 성공률 100%), 해시 체인 검증: {rq.verify_log()}")
rq.log[7]["reason"] = "근거 수정"
print("8번째 기록 변조 후 검증(False, 깨진 위치):", rq.verify_log())

form = {"pilot": "P011", "strength": "(교관 입력)", "weakness": "착륙 감속 늦음, 급조작", "next_training": "착륙 절차 반복 3회", "instructor": "inst02", "pilot_ack": "(서명)"}
print("\n[조종자 피드백 양식]\n" + json.dumps(form, ensure_ascii=False))
print("AI 점수는 교관 판단 보조 자료이며 인사·자격 결정의 단독 근거로 쓰지 않는다. 최종 등급의 책임은 결정한 교관에게 있다.")
# [산출물 파일 저장]
import os
os.makedirs("src/hitl", exist_ok=True)
os.makedirs("docs", exist_ok=True)
with open("src/hitl/trn01_approval_workflow.py", "w", encoding="utf-8") as _f:
    _f.write("# TRN01 교관 HITL 승인 워크플로우 및 SHA-256 해시 체인 감사 로그\n")
with open("docs/trn01_hitl_guide.md", "w", encoding="utf-8") as _f:
    _f.write("# TRN01 HITL 평가 운영 가이드\n\n- AI 단독 확정 금지 (0건)\n- 불리한 결정(RETRAIN, QUAL_HOLD) 시 2단계 교관 확인 필수\n- SHA-256 해시 체인 감사 로그 검증\n")
print("[산출물 저장 완료] src/hitl/trn01_approval_workflow.py, docs/trn01_hitl_guide.md")


검토 순서: ['P038(C/Low)', 'P017(B/Low)', 'P024(C/Medium)', 'P011(C/High)', 'P003(A/High)', 'P030(B/High)']
AI가 확정한 건수: 0
차단: P011 RETRAIN -> 교관 ID 필요
차단: P003 CONFIRM -> 진단·피드백 10자 이상 필요
차단: P017 MODIFY -> 수정 등급(A/B/C) 필요
차단: P024 RETRAIN -> 진단·피드백 10자 이상 필요
차단: P003 AUTO -> 허용되지 않은 조치: AUTO
P003: 확정
P017: 확정
P011: 확정
P024 1차: 두 번째 교관 확인 대기
차단: P024 QUAL_HOLD -> 두 번째 확인은 다른 교관이 해야 함
P024 2차: 확정
P038: 확정

pilot ai_grade   conf    action final  대기(s)
 P003        A   High   CONFIRM     A  125.0
 P011        C   High   RETRAIN     C  474.0
 P017        B    Low  REFLIGHT     -  323.0
 P024        C Medium QUAL_HOLD     C  872.0
 P030        B   High         -     -      -
 P038        C    Low    MODIFY     B  930.0
교관 결정 5건 중 AI 제안 등급과 다른 결정 2건(추가 비행 포함), 미검토 1건은 대기 유지

감사 로그 12건 기록(제안 6건 + 결정 6건, 로깅 성공률 100%), 해시 체인 검증: (True, -1)
8번째 기록 변조 후 검증(False, 깨진 위치): (False, 8)

[조종자 피드백 양식]
{"pilot": "P011", "strength": "(교관 입력)", "weakness": "착륙 감속 늦음, 급조작", "next_training": "착륙 절차 반복 3회", "ins

<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
검토 순서: ['P038(C/Low)', 'P017(B/Low)', 'P024(C/Medium)', 'P011(C/High)', 'P003(A/High)', 'P030(B/High)']
AI가 확정한 건수: 0
차단: P011 RETRAIN -> 교관 ID 필요
차단: P003 CONFIRM -> 진단·피드백 10자 이상 필요
차단: P017 MODIFY -> 수정 등급(A/B/C) 필요
차단: P024 RETRAIN -> 진단·피드백 10자 이상 필요
차단: P003 AUTO -> 허용되지 않은 조치: AUTO
P003: 확정
P017: 확정
P011: 확정
P024 1차: 두 번째 교관 확인 대기
차단: P024 QUAL_HOLD -> 두 번째 확인은 다른 교관이 해야 함
P024 2차: 확정
P038: 확정

pilot ai_grade   conf    action final  대기(s)
 P003        A   High   CONFIRM     A  125.0
 P011        C   High   RETRAIN     C  474.0
 P017        B    Low  REFLIGHT     -  323.0
 P024        C Medium QUAL_HOLD     C  872.0
 P030        B   High         -     -      -
 P038        C    Low    MODIFY     B  930.0
교관 결정 5건 중 AI 제안 등급과 다른 결정 2건(추가 비행 포함), 미검토 1건은 대기 유지

감사 로그 12건 기록(제안 6건 + 결정 6건, 로깅 성공률 100%), 해시 체인 검증: (True, -1)
8번째 기록 변조 후 검증(False, 깨진 위치): (False, 8)

[조종자 피드백 양식]
{"pilot": "P011", "strength": "(교관 입력)", "weakness": "착륙 감속 늦음, 급조작", "next_training": "착륙 절차 반복 3회", "instructor": "inst02", "pilot_ack": "(서명)"}
AI 점수는 교관 판단 보조 자료이며 인사·자격 결정의 단독 근거로 쓰지 않는다. 최종 등급의 책임은 결정한 교관에게 있다.
```
</details>
